# DistilBERT

This notebook fine-tunes `distilbert-base-uncased` using  `review_text_translated` from `data/master_clean.csv`.

### Imports and Config

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
FORCE_CPU = False
if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [2]:
from pathlib import Path
import inspect
import json
import random

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "model_training":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "master_clean.csv"
OUTPUT_DIR = PROJECT_ROOT / "model_training" / "distilbert_outputs" / "translated_english"
MODEL_SAVE_DIR = PROJECT_ROOT / "model_training" / "distilbert_sentiment_model" / "translated_english"
RESULTS_PATH = PROJECT_ROOT / "model_training" / "distilbert_experiment_results.csv"

MODEL_NAME = "distilbert-base-uncased"
TEXT_COL = "review_text_translated"
LABEL_COL = "sentiment"
SPLIT_COL = "split"

LABEL_NAMES = ["negative", "neutral", "positive"]
LABEL2ID = {label: idx for idx, label in enumerate(LABEL_NAMES)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

MAX_LENGTH = 160
SEED = 42
NUM_EPOCHS = 4
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() and not FORCE_CPU else "cpu"
print(f"Using device: {device}")

c:\Users\pangt\anaconda\envs\nlpbert\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


### Load Data

In [3]:
df = pd.read_csv(DATA_PATH)

required_cols = {TEXT_COL, LABEL_COL, SPLIT_COL}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

df = df.dropna(subset=[TEXT_COL, LABEL_COL, SPLIT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].astype(str).str.lower().str.strip()
df = df[df[TEXT_COL] != ""].copy()

unknown_labels = sorted(set(df[LABEL_COL]) - set(LABEL_NAMES))
if unknown_labels:
    raise ValueError(f"Unknown labels found: {unknown_labels}")

df["label"] = df[LABEL_COL].map(LABEL2ID)

train_df = df[df[SPLIT_COL] == "train"].reset_index(drop=True)
val_df = df[df[SPLIT_COL] == "val"].reset_index(drop=True)
test_df = df[df[SPLIT_COL] == "test"].reset_index(drop=True)

print("Rows:", {"train": len(train_df), "val": len(val_df), "test": len(test_df)})
display(df.groupby([SPLIT_COL, LABEL_COL]).size().unstack(fill_value=0).reindex(columns=LABEL_NAMES))
df[[TEXT_COL, LABEL_COL, SPLIT_COL]].head()

Rows: {'train': 428, 'val': 92, 'test': 92}


sentiment,negative,neutral,positive
split,,,
test,29,16,47
train,134,74,220
val,29,15,48


,review_text_translated,sentiment,split
0,item arrived well cute power good sound even l...,neutral,test
1,performance good quality superb small powerful...,neutral,val
2,bluetooth keeps disconnecting use aux noise fe...,negative,train
3,quality matches price charging cable right hah...,neutral,test
4,quality sound quality supply not even bother c...,negative,test


### Dataset, Metrics, and Weighted Trainer

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        weights = self.class_weights.to(logits.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "neutral_f1": f1_score(labels, preds, labels=[LABEL2ID["neutral"]], average="macro", zero_division=0),
    }

train_dataset = ReviewDataset(train_df[TEXT_COL].tolist(), train_df["label"].tolist())
val_dataset = ReviewDataset(val_df[TEXT_COL].tolist(), val_df["label"].tolist())
test_dataset = ReviewDataset(test_df[TEXT_COL].tolist(), test_df["label"].tolist())

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["label"].to_numpy(),
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights.tolist())

Class weights: [1.0646766424179077, 1.9279279708862305, 0.6484848260879517]


c:\Users\pangt\anaconda\envs\nlpbert\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Train

In [5]:
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

training_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "num_train_epochs": NUM_EPOCHS,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": 0.01,
    "logging_dir": str(OUTPUT_DIR / "logs"),
    "logging_steps": 20,
    "save_strategy": "epoch",
    "save_total_limit": 1,
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "report_to": "none",
    "seed": SEED,
}

training_arg_names = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in training_arg_names:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
 10%|▉         | 21/212 [00:06<00:22,  8.42it/s]

{'loss': 1.0926, 'grad_norm': 2.085676908493042, 'learning_rate': 1.8113207547169813e-05, 'epoch': 0.37}


 19%|█▉        | 41/212 [00:08<00:20,  8.31it/s]

{'loss': 1.0452, 'grad_norm': 4.216739654541016, 'learning_rate': 1.6226415094339625e-05, 'epoch': 0.75}


                                                
 25%|██▌       | 53/212 [00:10<00:18,  8.49it/s]

...

                                                 
100%|██████████| 212/212 [00:31<00:00,  8.22it/s]

{'eval_loss': 0.6345254778862, 'eval_accuracy': 0.8043478260869565, 'eval_macro_f1': 0.6719298245614035, 'eval_weighted_f1': 0.7732265446224256, 'eval_neutral_f1': 0.3157894736842105, 'eval_runtime': 0.1596, 'eval_samples_per_second': 576.502, 'eval_steps_per_second': 75.196, 'epoch': 3.96}


100%|██████████| 212/212 [00:33<00:00,  6.40it/s]

{'train_runtime': 33.1497, 'train_samples_per_second': 51.644, 'train_steps_per_second': 6.395, 'train_loss': 0.7230422271872466, 'epoch': 3.96}


TrainOutput(global_step=212, training_loss=0.7230422271872466, metrics={'train_runtime': 33.1497, 'train_samples_per_second': 51.644, 'train_steps_per_second': 6.395, 'total_flos': 39053741421888.0, 'train_loss': 0.7230422271872466, 'epoch': 3.9626168224299065})

### Evaluate

In [7]:
val_metrics = trainer.evaluate(val_dataset)
test_metrics = trainer.evaluate(test_dataset)
test_predictions = trainer.predict(test_dataset)

y_true = test_predictions.label_ids
y_pred = np.argmax(test_predictions.predictions, axis=1)
report = classification_report(
    y_true,
    y_pred,
    target_names=LABEL_NAMES,
    output_dict=True,
    zero_division=0,
)

print("Validation metrics:")
print(val_metrics)
print("Test metrics:")
print(test_metrics)
print("Test classification report:")
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, zero_division=0))

cm = pd.DataFrame(
    confusion_matrix(y_true, y_pred),
    index=[f"actual_{label}" for label in LABEL_NAMES],
    columns=[f"pred_{label}" for label in LABEL_NAMES],
)
display(cm)

100%|██████████| 12/12 [00:00<00:00, 76.31it/s]


Validation metrics:
{'eval_loss': 0.6345254778862, 'eval_accuracy': 0.8043478260869565, 'eval_macro_f1': 0.6719298245614035, 'eval_weighted_f1': 0.7732265446224256, 'eval_neutral_f1': 0.3157894736842105, 'eval_runtime': 0.367, 'eval_samples_per_second': 250.688, 'eval_steps_per_second': 32.698, 'epoch': 3.9626168224299065}
Test metrics:
{'eval_loss': 0.6673896908760071, 'eval_accuracy': 0.75, 'eval_macro_f1': 0.6242617431142021, 'eval_weighted_f1': 0.7249688618184698, 'eval_neutral_f1': 0.25, 'eval_runtime': 0.1533, 'eval_samples_per_second': 600.002, 'eval_steps_per_second': 78.261, 'epoch': 3.9626168224299065}
Test classification report:
              precision    recall  f1-score   support

    negative       0.72      0.79      0.75        29
     neutral       0.38      0.19      0.25        16
    positive       0.83      0.91      0.87        47

    accuracy                           0.75        92
   macro avg       0.64      0.63      0.62        92
weighted avg       0.71   

,pred_negative,pred_neutral,pred_positive
actual_negative,23,3,3
actual_neutral,7,3,6
actual_positive,2,2,43


### Save Model and Result

In [8]:
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_SAVE_DIR))
tokenizer.save_pretrained(str(MODEL_SAVE_DIR))

with open(MODEL_SAVE_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"label2id": LABEL2ID, "id2label": ID2LABEL}, f, indent=2)

result = {
    "experiment": "translated_english",
    "text_col": TEXT_COL,
    "model_name": MODEL_NAME,
    "val_accuracy": val_metrics["eval_accuracy"],
    "val_macro_f1": val_metrics["eval_macro_f1"],
    "val_neutral_f1": val_metrics["eval_neutral_f1"],
    "test_accuracy": test_metrics["eval_accuracy"],
    "test_macro_f1": test_metrics["eval_macro_f1"],
    "test_neutral_f1": report["neutral"]["f1-score"],
    "model_dir": str(MODEL_SAVE_DIR),
}

results_df = pd.DataFrame([result])
results_df.to_csv(RESULTS_PATH, index=False)

print(f"Saved model to: {MODEL_SAVE_DIR}")
print(f"Saved result to: {RESULTS_PATH}")
display(results_df)

Saved model to: c:\Users\pangt\Downloads\ecom-reviews-sentiment-analysis\model_training\distilbert_sentiment_model\translated_english
Saved result to: c:\Users\pangt\Downloads\ecom-reviews-sentiment-analysis\model_training\distilbert_experiment_results.csv


,experiment,text_col,model_name,val_accuracy,val_macro_f1,val_neutral_f1,test_accuracy,test_macro_f1,test_neutral_f1,model_dir
0,translated_english,review_text_translated,distilbert-base-uncased,0.804348,0.67193,0.315789,0.75,0.624262,0.25,c:\Users\pangt\Downloads\ecom-reviews-sentimen...


### Quick Prediction Test

In [9]:
def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))
    return {
        "label": ID2LABEL[pred_id],
        "confidence": float(probs[pred_id]),
        "probabilities": {ID2LABEL[i]: float(prob) for i, prob in enumerate(probs)},
    }

predict_sentiment("not good quality and delivery was slow")

{'label': 'negative',
 'confidence': 0.46575990319252014,
 'probabilities': {'negative': 0.46575990319252014,
  'neutral': 0.44477757811546326,
  'positive': 0.08946250379085541}}